In [ ]:

import getpass
from dotenv import load_dotenv
import pydantic
from crewai import LLM, Agent, Task, Crew
from com.example.agentic.agent.LLMManager import LLMManager


llm = LLMManager.create_llm('openai')
llm.call('Hello')

In [1]:
from crewai_tools import RagTool
from com.example.agentic.tools.config import _rag_tool_config
from crewai.rag.config.utils import set_rag_config, get_rag_client, clear_rag_config
from chromadb.utils.embedding_functions.openai_embedding_function import (
        OpenAIEmbeddingFunction,
    )

# Create a RAG tool with custom configuration
# rag_tool = RagTool(config=_rag_tool_config, summarize=True)

# ChromaDB (default)
from crewai.rag.chromadb.config import ChromaDBConfig
set_rag_config(ChromaDBConfig(
    embedding_function = OpenAIEmbeddingFunction(
            api_key=os.getenv("OPENAI_API_KEY"),
            api_base="http://localhost:11434/v1/",
            model_name="nomic-embed-text:latest",
            api_version="v1",
            api_type="ollama",
            api_key_env_var="OPENAI_API_KEY",
        )
))

chromadb_client = get_rag_client()

# Example operations (same API for any provider)
client = chromadb_client  # or chromadb_client

#client.create_collection(collection_name="pdf-documents")

In [ ]:
import os 
from com.example.agentic.loader.LoadManager import LoadManager
# 
#client.create_collection(collection_name="pdf-documents")

work_dir = os.getenv("WORK_DIR")
_pdf_documents = [
    # f"{work_dir}/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf",
    # f"{work_dir}/docs/pdf/Advanced_Microservices.pdf",
    # f"{work_dir}/docs/pdf/Building Microservices - Designing Fine-Grained Systems.pdf",
    # f"{work_dir}/docs/pdf/Clean Architecture A Craftsman Guide to Software Structure and Design.pdf",
    # f"{work_dir}/docs/pdf/Data Science.pdf",
    f"{work_dir}/docs/pdf/Designing_Data_Intensive_Applications.pdf",
    f"{work_dir}/docs/pdf/Enterprise Integration Patterns - Designing, Building And Deploying Messaging.pdf",
    # f"{work_dir}/docs/pdf/Kafka_Streams_In_Action.pdf",
    # f"{work_dir}/docs/pdf/Microservices Best_Practices_for_Java.pdf",
    # f"{work_dir}/docs/pdf/Microservices_From_Day_one.pdf",
    # f"{work_dir}/docs/pdf/Microservices From Design to Deploy.pdf",
    # f"{work_dir}/docs/pdf/Microservices Patterns.pdf",
    # f"{work_dir}/docs/pdf/ML_book.pdf",
    # f"{work_dir}/docs/pdf/Software Architecture Patterns.pdf",
    # f"{work_dir}/docs/pdf/System Design-interview-an-insiders.pdf"
]
import uuid
from crewai.rag.types import BaseRecord
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10, length_function=len, separators=["\n\n", "\n", " ", ""])

for _pdf_doc in _pdf_documents :
    # Prepare data for ChromaDB
    _documents = LoadManager.from_pdfs([_pdf_doc])
    _documents_chunks = splitter.split_documents(_documents)
    if len(_documents_chunks):
        _docs_to_store = []
        for i,_d in enumerate(_documents_chunks) :
            _doc = dict()

            doc_id = f"pdf_{uuid.uuid4().hex[:8]}_{i}"
            _doc["id"] = doc_id
            _doc['content'] = _d.page_content

            # Prepare metadata
            _metadata = dict(_d.metadata)
            _metadata['doc_index'] = i
            _metadata['content_length'] = len(_d.page_content)
            #_metadata['content'] = _d.page_content
            _doc["metadata"] = _metadata
            
            #print(_doc)

            # Document content
            _docs_to_store.append(_doc)
            
            #_d.content = _d.page_content

        client.add_documents(
            collection_name="pdf-documents",
            documents=_docs_to_store
        )
    else :
        print("document chunks not provided.")

[DEBUG] Found 1 page : ['/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/pdf/Data Science.pdf']
[DEBUG] Loading pdf document from : /home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/pdf/Data Science.pdf
[DEBUG] Loaded 9 PDF document from /home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/pdf/Data Science.pdf
document chunks not provided.


In [ ]:
# Add content from a file
#rag_tool.add(data_type="pdf_file", path=f"{work_dir}/docs/pdf/Advanced_Microservices.pdf")

In [21]:
results = client.search(collection_name="pdf-documents", query="Software Architecture", limit=10)
for _result in results :
    print(_result)
    
#clear_rag_config()  # optional reset

{'id': '95ea3dde18f96d9563fe13609ca47851574878ddf48832dbc3949c20f15a7fcb', 'content': 'Protocols �����������������������������������������������������������������������������������������������������������������11', 'metadata': {'content_length': 125, 'page_label': 'v', 'creationdate': '2017-06-10T19:38:51+05:30', 'page': 4, 'creator': 'Adobe InDesign CS6 (Windows)', 'moddate': '2017-06-10T19:51:37+05:30', 'doc_index': 17, 'source': '/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/pdf/Advanced_Microservices.pdf', 'producer': 'Adobe PDF Library 10.0.1', 'total_pages': 193}, 'score': 1.0}
{'id': '865f530eede9323838c38a7ca5812d3c8d43069779b35167865bcaf060a6739f', 'content': 'Patches Welcome ��������������������������������������������������������������������������������169\nReferences �������������������������������������������������������������������������������������171\nIndex ����������������������������������������������������������������������������������������������175',

In [ ]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage

class CustomKnowledgeStorage(KnowledgeStorage):

    def __init__(self, persist_directory: str, embedder=None, collection_name=None):
        self.persist_directory = persist_directory
        self.embedder = embedder
        self.collection_name = collection_name
        super().__init__(embedder=embedder, collection_name=collection_name)
    
    def initialize_knowledge_storage(self):
        pass

In [ ]:
from crewai.tools import tool
from crewai_tools import PDFSearchTool
from com.example.agentic.tools.config import _tool_config, _rag_tool_config, _embedder_config_openai
#pydantic.SkipValidation

# 1. Initialize the tool
# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
#@tool

_pdf_tool = PDFSearchTool(
    pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
    config=_rag_tool_config
)
#_pdf_tool.add(pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf')

In [ ]:
results = _pdf_tool.run(query="Spark & Kafka")
results

In [ ]:
from datetime import datetime
from crewai import Crew, Agent, Task, Process
from crewai import context
from emoji import config
from com.example.agentic.tools.config import _tool_config, _embedder_config_openai

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(run_id)

from crewai_tools import FileReadTool

# Initialize tool
file_read_tool = FileReadTool(file_path='/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt')
#file_read_tool.run()

# 3. Agent and Task Definition
text_specialist = Agent(
    role='Text File Reader',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[file_read_tool],
    #llm=llm
)

# 2. Define the Agent
resume_analyst = Agent(
    role='As a resume analyst',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[],
    allow_delegation=False,
    verbose=True,
    #llm=llm
)

# 3. Define the Task
read_task = Task(
    description='Read text content',
    expected_output='text contain in the file.',
    agent=text_specialist,
    output_file=f'outputs/L004/read_task_{run_id}.txt'
)

extract_task = Task(
    description='review the content that you receive and make sure you extract skills, experience and work location',
    expected_output='A json string contain skills, experience and work location and list of domain knowledge',
    agent=text_specialist,
    context=[read_task],
    output_file=f'outputs/L004/extract_task_{run_id}.json'
)

# 4. Run the Crew
crew = Crew(
    agents=[text_specialist], 
    tasks=[read_task,extract_task],
    process=Process.sequential,
    verbose=True,
    #planning=True,
    #planning_llm="openai/llama3.2:1b-instruct-q8_0"
)
result = crew.kickoff()
print(result)